In [ ]:
from datetime import date
date.today().strftime("%Y-%m-%d")

In [ ]:
import os
os.getlogin()

In [ ]:
import sys
print(sys.version)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
from scipy.stats import median_abs_deviation
import re
import platform


sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)

In [ ]:
os.environ['CONDA_DEFAULT_ENV'] # conda env

In [ ]:
platform.platform()

In [ ]:
# load R environmekeysif os.name == 'nt': # only for Windows
if os.name == 'nt': # only for Windows
    os.environ["R_HOME"] = r"C:\Programme\R\R-4.3.1"
    
import anndata2ri
import logging

import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
import muon as mu

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [ ]:
sc.settings.verbosity = 3

In [ ]:

main_dir="\\\isdsynnas.srv.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\\" # Win


In [ ]:
analysis_dir = "/snRNAseq_Results_Leiden2/"

In [ ]:
#Load all samples
#Into dict

all_dir=main_dir+analysis_dir
x=os.listdir(os.path.join(all_dir))
x

full_paths = [os.path.join(all_dir, item) for item in x]
full_paths

In [ ]:
# load all samples as adata into dict (keys=samplename, values=adata objects)
#mdata = {p.split(".")[0]:mu.read_10x_h5(os.path.join(all_dir,p,"outs","filtered_feature_bc_matrix.h5")) for p in x} # file name as input
mdata = {p.split("/")[2]:mu.read_10x_h5(os.path.join(p,"outs","filtered_feature_bc_matrix.h5")) for p in full_paths} # full path as in

In [ ]:
for key in mdata.keys():
    mdata[key].var_names_make_unique()

In [ ]:
mdata.keys()

In [ ]:
mdata.values()

In [ ]:
# read raw matrix for ambient RNA correction
adata_raw = {p.split("/")[2]:sc.read_10x_h5(os.path.join(p,"outs","raw_feature_bc_matrix.h5")) for p in full_paths}
#adata_raw["mo00-WT"] = adata_raw["sample00"]
#del adata_raw["sample00"]

In [ ]:
adata_raw

In [ ]:
#Subset RNA

In [ ]:
adatas = {}
for key in mdata.keys():
    adatas[key] = mdata[key].mod['rna']
    adatas[key]

In [ ]:
adatas

In [ ]:
for key in adatas.keys():
    adatas[key].layers

In [ ]:
# genes with highest fraction of counts per cell
for key in adatas.keys():
    print(key)
    sc.pl.highest_expr_genes(adatas[key], n_top=20, )

In [ ]:
#QC


In [ ]:
for key in adatas.keys():
    # mitochondrial genes
    adatas[key].var["mt"] = adatas[key].var_names.str.startswith("mt-")
    # ribosomal genes
    adatas[key].var["ribo"] = adatas[key].var_names.str.startswith(("Rps", "Rpl"))
    # hemoglobin genes.
    adatas[key].var["hb"] = adatas[key].var_names.str.contains(("^Hb[^(P)]"))

In [ ]:
for key in adatas.keys():
    sc.pp.calculate_qc_metrics(
        adatas[key], qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
    )

In [ ]:
for key in adatas.keys():
    print('-'*100)
    print(key)
    print('Genes per cells (mean): ',np.rint(np.mean(adatas[key].obs.n_genes_by_counts)))
    print('Counts per cells (mean): ',np.rint(np.mean(adatas[key].obs.total_counts)))
    print('Counts per cells (median): ',np.rint(np.median(adatas[key].obs.total_counts)))
    sc.pl.violin(adatas[key], ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb'], jitter=0.4, multi_panel=True, size=0.2)
    #sns.displot(adatas[key].obs["total_counts"], bins=100, kde=False)

In [ ]:
for key in adatas.keys():
    print(np.median(adatas[key].obs.total_counts))

In [ ]:
for key in adatas.keys():
    sns.displot(adatas[key].obs["total_counts"], bins=100, kde=False).set(title=key)

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pl.scatter(adatas[key], x='total_counts', y='pct_counts_mt', color="pct_counts_mt")
    sc.pl.scatter(adatas[key], x='total_counts', y='n_genes_by_counts', color="pct_counts_mt")

## Filtering low quality reads
The first step in quality control is to remove low-quality reads from the dataset. When a cell has a low number of detected genes, a low count depth and a high fraction of mitochondrial counts it might have a broken membrane which can indicate a dying cell. As these cells are usually not the main target of our analysis and might distort our downstream analysis, we are removing them during quality control. For identifying them, we define cell quality control (QC) threshold. Cell QC is typically performed on the following three QC covariates:

The number of counts per barcode (count depth)
The number of genes per barcode
The fraction of counts from mitochondrial genes per barcode
In cell QC these covariates are filtered via thresholding as they might correspond to dying cells. As indicated, they might reflect cells with broken membranes whose cytoplasmic mRNA has leaked out and therefore only the mRNA in the mitochondria is still present. These cells might then show a low count depth, few detected genes and a high fraction of mitochondrial reads. It is, however, crucial to consider the three QC covariates jointly as otherwise it might lead to misinterpretation of cellular signals. Cells with a relatively high fraction of mitochondrial counts might for example be involved in respiratory processes and should not be filtered out. Whereas, cells with low or high counts might correspond to quiescent cell populations or cells larger in size. It is therefore preferred to consider multiple covariates when thresholding decisions on single covariates are made. In general, it is advised to exclude fewer cells and be as permissive as possible to avoid filtering out viable cell populations or small sub-populations.

QC on only a few or small datasets is often performed in a manual fashion by looking at the distribution of different QC covariates and identifying outlier which will be filtered afterwards. However, as datasets grow in size this task is becoming more and more time-consuming and it might be worth considering automatic thresholding via MAD (median absolute deviations). The MAD is given by  𝑀𝐴𝐷=𝑚𝑒𝑑𝑖𝑎𝑛(|𝑋𝑖−𝑚𝑒𝑑𝑖𝑎𝑛(𝑋)|)
  with  𝑋𝑖
  being the respective QC metric of an observation and describes a robust statistic of the variability of the metric. Similar to {cite}qc:germain_pipecomp_2020, we mark cells as outliers if they differ by 5 MADs which is a relatively permissive filtering strategy. We want to highlight that it might be reasonable to re-assess the filtering after annotation of cells.

In QC, the first step is to calculate the QC covariates or metric. We compute these using the scanpy function sc.pp.calculate_qc_metrics, which can also calculate the proportions of counts for specific gene populations. We therefore define mitochondrial, ribosomal and hemoglobin genes. It is important to note that mitochondrial counts are annotated either with the prefix "mt-" or "MT-'' depending on the species considered in the dataset. As mentioned, the dataset used in this notebook is human bone marrow, so mitochondrial counts are annotated with the prefix "MT-". For mouse datasets, the prefix is usually lower case, so "mt-".

In [ ]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
for key in adatas.keys():
    adatas[key].obs["outlier"] = (
        is_outlier(adatas[key], "log1p_total_counts", 5)
        | is_outlier(adatas[key], "log1p_n_genes_by_counts", 5)
        | is_outlier(adatas[key], "pct_counts_in_top_20_genes", 5)
    )
    print(key)
    print(adatas[key].obs.outlier.value_counts())

In [ ]:
for key in adatas.keys():
    adatas[key].obs["mt_outlier"] = is_outlier(adatas[key], "pct_counts_mt", 3) | (
        adatas[key].obs["pct_counts_mt"] > 15
    )
    print(key)
    print(adatas[key].obs.mt_outlier.value_counts())

In [ ]:
for key in adatas.keys():
    print(key)
    print(f"Total number of cells: {adatas[key].n_obs}")
    adatas[key] = adatas[key][(~adatas[key].obs.outlier) & (~adatas[key].obs.mt_outlier)].copy()
    print(f"Number of cells after filtering of low quality cells: {adatas[key].n_obs}")

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pl.scatter(adatas[key], x='total_counts', y='pct_counts_mt', color="pct_counts_mt")
    sc.pl.scatter(adatas[key], x='total_counts', y='n_genes_by_counts', color="pct_counts_mt")

In [ ]:
%%R
library(SoupX)

In [ ]:
key = list(adatas.keys())[0]
key

In [ ]:
print(key)
adata_pp = adatas[key].copy()
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="soupx_groups")

# Preprocess variables for SoupX
soupx_groups = adata_pp.obs["soupx_groups"]
    
# delete adata copy
del adata_pp
    
# prepare for SoupX
cells = adatas[key].obs_names
genes = adatas[key].var_names
data = adatas[key].X.T
    
#adata_raw = sc.read_10x_h5(os.path.join(all_dir,key,"outs","raw_feature_bc_matrix.h5"))
adata_raw[key].var_names_make_unique()
data_tod = adata_raw[key].X.T

In [ ]:
%%R -i data -i data_tod -i genes -i cells -i soupx_groups -o out 

# specify row and column names of data
rownames(data) = genes
colnames(data) = cells
# ensure correct sparse format for table of counts and table of droplets
data <- as(data, "sparseMatrix")
data_tod <- as(data_tod, "sparseMatrix")

# Generate SoupChannel Object for SoupX 
sc = SoupChannel(data_tod, data, calcSoupProfile = FALSE)

# Add extra meta data to the SoupChannel object
soupProf = data.frame(row.names = rownames(data), est = rowSums(data)/sum(data), counts = rowSums(data))
sc = setSoupProfile(sc, soupProf)
# Set cluster information in SoupChannel
sc = setClusters(sc, soupx_groups)

# Estimate contamination fraction
sc  = autoEstCont(sc, doPlot=FALSE)
# Infer corrected table of counts and rount to integer
out = adjustCounts(sc, roundToInt = TRUE)

In [ ]:
adatas[key].layers["counts"] = adatas[key].X
adatas[key].layers["soupX_counts"] = out.T
adatas[key].X = adatas[key].layers["soupX_counts"]

In [ ]:
print(adatas[key].X)
print(adatas[key].layers["soupX_counts"])

In [ ]:
print(f"Total number of genes: {adatas[key].n_vars}")
# Min 20 cells - filters out 0 count genes
sc.pp.filter_genes(adatas[key], min_cells=20)
print(f"Number of genes after cell filter: {adatas[key].n_vars}")

In [ ]:
del data
del data_tod
del genes
del cells
del soupx_groups
del out

In [ ]:
key = list(adatas.keys())[1]
key

In [ ]:
print(key)
adata_pp = adatas[key].copy()
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="soupx_groups")

# Preprocess variables for SoupX
soupx_groups = adata_pp.obs["soupx_groups"]
    
# delete adata copy
del adata_pp
    
# prepare for SoupX
cells = adatas[key].obs_names
genes = adatas[key].var_names
data = adatas[key].X.T
    
#adata_raw = sc.read_10x_h5(os.path.join(all_dir,key,"outs","raw_feature_bc_matrix.h5"))
adata_raw[key].var_names_make_unique()
data_tod = adata_raw[key].X.T

In [ ]:
%%R -i data -i data_tod -i genes -i cells -i soupx_groups -o out 

# specify row and column names of data
rownames(data) = genes
colnames(data) = cells
# ensure correct sparse format for table of counts and table of droplets
data <- as(data, "sparseMatrix")
data_tod <- as(data_tod, "sparseMatrix")

# Generate SoupChannel Object for SoupX 
sc = SoupChannel(data_tod, data, calcSoupProfile = FALSE)

# Add extra meta data to the SoupChannel object
soupProf = data.frame(row.names = rownames(data), est = rowSums(data)/sum(data), counts = rowSums(data))
sc = setSoupProfile(sc, soupProf)
# Set cluster information in SoupChannel
sc = setClusters(sc, soupx_groups)

# Estimate contamination fraction
sc  = autoEstCont(sc, doPlot=FALSE)
# Infer corrected table of counts and rount to integer
out = adjustCounts(sc, roundToInt = TRUE)

In [ ]:
adatas[key].layers["counts"] = adatas[key].X
adatas[key].layers["soupX_counts"] = out.T
adatas[key].X = adatas[key].layers["soupX_counts"]

In [ ]:
print(f"Total number of genes: {adatas[key].n_vars}")
# Min 20 cells - filters out 0 count genes
sc.pp.filter_genes(adatas[key], min_cells=20)
print(f"Number of genes after cell filter: {adatas[key].n_vars}")

In [ ]:
del data
del data_tod
del genes
del cells
del soupx_groups
del out

(rna:doublet-detection)=
## Doublet Detection

Doublets are defined as two cells that are sequenced under the same cellular barcode, for example, if they were captured in the same droplet. That's why we used the term "barcode" instead of "cell" until now. A doublet is called homotypic if it is formed by the same cell type (but from different individuals) and heterotypic otherwise. Homotypic doublets are not necessarily identifiable from count matrices and are often considered innocuous as they can be identified with cell hashing or SNPs. Hence, their identification is not the main goal of the doublet detection methods. 

Doublets formed from different cell types or states are called heterotypic. Their identification is crucial as they are most likely misclassified and can lead to distorted downstream analysis steps. Hence, doublet detection and removal are typically initial preprocessing steps. Doublets can be either identified through their high number of reads and detected features, or with methods that create artificial doublets and compare these with the cells present in the dataset. Doublet detection methods are computationally efficient and there exist several software packages for this task. 

{cite}`xi_benchmarking_2021` benchmarked nine different doublet detection methods and assessed their performance with respect to computational efficiency and doublet detection accuracy. They additionally assessed scDblFinder in an addendum to their benchmark which achieved the highest doublet detection accuracy and a good computational efficiency and stability {cite}`Xi_2021_addendum`.

In this tutorial, we will showcase the scDblFinder R package. scDblFinder randomly selects two droplets and creates artificial doublets from those by averaging their gene expression profiles. The doublet score is then defined as the fraction of artificial doublets in the k-nearest neighbor graph of each droplet in the principal component space.

:::{figure-md} Doublet detection

<img src="../_static/images/preprocessing_visualization/doublet_detection.jpeg" alt="Doublet detection overview" class="bg-primary mb-1" width="800px">

Doublets are droplets that contain more than one cell. Common doublet detection methods generate artificial doublets by randomly subsampling pairs of cells and averaging their gene expression profile to obtain doublet counts. These artificial doublets are projected together with the remaining calls into a lower dimensional principal component space. The doublet detection method computes a doublet score based on the number of artificial doublet neighbors in the k nearest neighbor graph.

:::

We first load some additional python and R packages.

In [ ]:
%%R
library(Seurat)
library(scater)
library(scDblFinder)
library(BiocParallel)

In [ ]:
key = list(adatas.keys())[0]
key

In [ ]:
data_mat = adatas[key].X.T

We can now launch the doublet detection by using data_mat as input to scDblFinder within a SingleCellExperiment. scBblFinder adds several columns to the colData of sce. Three of them might be interesting for the analysis:

sce$scDblFinder.score: the final doublet score (the higher the more likely that the cell is a doublet)

sce$scDblFinder.ratio : the ratio of artificial doublets in the cell's neighborhood

sce$scDblFinder.class : the classification (doublet or singlet)

We will only output the class argument and store it in the AnnData object in .obs . The other arguments can be added to the AnnData object similarly.

In [ ]:
%%R -i data_mat -o doublet_score -o doublet_class

set.seed(123)
sce = scDblFinder(
    SingleCellExperiment(
        list(counts=data_mat),
    ) 
)
doublet_score = sce$scDblFinder.score
doublet_class = sce$scDblFinder.class

In [ ]:
#scDblFinder outputs a class with the classification Singlet (1) and Doublet (2). We add this to our AnnData object in .obs.

adatas[key].obs["scDblFinder_score"] = doublet_score
adatas[key].obs["scDblFinder_class"] = doublet_class
adatas[key].obs.scDblFinder_class.value_counts()

In [ ]:
print(f"Total number of cells: {adatas[key].n_obs}")
#adatas[key] = adatas[key][adatas[key].obs.scDblFinder_class != 2].copy()
#print(f"Number of cells after filtering of low quality cells: {adatas[key].n_obs}")

In [ ]:
del data_mat
del doublet_score
del doublet_class

In [ ]:
key = list(adatas.keys())[1]
key

In [ ]:
data_mat = adatas[key].X.T

In [ ]:
%%R -i data_mat -o doublet_score -o doublet_class

set.seed(123)
sce = scDblFinder(
    SingleCellExperiment(
        list(counts=data_mat),
    ) 
)
doublet_score = sce$scDblFinder.score
doublet_class = sce$scDblFinder.class

In [ ]:
adatas[key].obs["scDblFinder_score"] = doublet_score
adatas[key].obs["scDblFinder_class"] = doublet_class
adatas[key].obs.scDblFinder_class.value_counts()

In [ ]:
del data_mat
del doublet_score
del doublet_class

##Normalisation, logarithmization
[Heumos2023]: The difference in gene expression between cells in the count data might simply be due to sampling effects. This means that the dataset and therefore the count matrix still contains widely varying variance terms. Analyzing the dataset is often challenging as many statistical methods assume data with uniform variance structure.

The preprocessing step of “normalization” aims to adjust the raw counts in the dataset for variable sampling effects by scaling the observable variance to a specified range. In the best practise paper, three different normalization techniques are introduced: the shifted logarithm transformation, scran normalization and analytic approximation of Pearson residuals. The shifted logarithm works beneficial for stabilizing variance for subsequent dimensionality reduction and identification of differentially expressed genes. Scran was extensively tested and used for batch correction tasks and analytic Pearson residuals are well suited for selecting biologically variable genes and identification of rare cell types.

For use in Tangram, preprocessing should be as similar as possible between spatial and single-cell data. Also see https://github.com/broadinstitute/Tangram/issues/3.
Hence, normalize and logarithmise with default settings, but don't scale.

Check if data is normalized:

In [ ]:
for key in adatas.keys():
    print(key)
    print(adatas[key].layers)

In [ ]:
for key in adatas.keys():
    scales_counts = sc.pp.normalize_total(adatas[key], target_sum=None, inplace=False)
    adatas[key].layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)

In [ ]:
# before vs after
for key in adatas.keys():
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    p1 = sns.histplot(adatas[key].obs["total_counts"], bins=100, kde=False, ax=axes[0])
    axes[0].set_title(key+": Total counts")
    p2 = sns.histplot(adatas[key].layers["log1p_norm"].sum(1), bins=100, kde=False, ax=axes[1])
    axes[1].set_title(key+": Shifted logarithm")
    plt.show()

[Heumos2023]: Single-cell RNA-seq datasets usually contain up to 30,000 genes and so far we only removed genes that are not detected in at least 20 cells. However, many of the remaining genes are not informative and contain mostly zero counts. Therefore, a standard preprocessing pipeline involves the step of feature selection which aims to exclude uninformative genes which might not represent meaningful biological variation across samples.

Traditional approaches and pipelines either compute the coefficient of variation (highly variable genes) or the average expression level (highly expressed genes) of all genes to obtain 500-2000 selected genes and use these features for their downstream analysis steps. However, these methods are highly sensitive to the normalization technique used before.

Germain et al. instead proposes to use deviance for feature selection which works on raw counts [Germain et al., 2020]. Deviance can be computed in closed form and quantifies whether genes show a constant expression profile across cells as these are not informative. Genes with constant expression are described by a multinomial null model, they are approximated by the binomial deviance. Highly informative genes across cells will have a high deviance value which indicates a poor fit by the null model (i.e., they don’t show constant expression across cells). According to the deviance values, the method then ranks all genes and obtains only highly deviant genes.

We use the R package scry to compute deviance.

In [ ]:
%%R
library(scry)

In [ ]:
key = list(adatas.keys())[0]
key

In [ ]:
ro.globalenv["adata"] = adatas[key] # save the AnnData object in our R environment

In [ ]:
%%R
sce = devianceFeatureSelection(adata, assay="X")

In [ ]:
binomial_deviance = ro.r("rowData(sce)$binomial_deviance").T

In [ ]:
idx = binomial_deviance.argsort()[-4000:]
mask = np.zeros(adatas[key].var_names.shape, dtype=bool)
mask[idx] = True

adatas[key].var["highly_deviant"] = mask
adatas[key].var["binomial_deviance"] = binomial_deviance

In [ ]:
key = list(adatas.keys())[1]
key

In [ ]:
ro.globalenv["adata"] = adatas[key] # save the AnnData object in our R environment

In [ ]:
%%R
sce = devianceFeatureSelection(adata, assay="X")

In [ ]:
binomial_deviance = ro.r("rowData(sce)$binomial_deviance").T

In [ ]:
idx = binomial_deviance.argsort()[-4000:]
mask = np.zeros(adatas[key].var_names.shape, dtype=bool)
mask[idx] = True

adatas[key].var["highly_deviant"] = mask
adatas[key].var["binomial_deviance"] = binomial_deviance

Dim Reduction
[Heumos2023]: Curse of dimensionality: The Curse of dimensionality was first brought up by R. Bellman [Bellman et al., 1957] and descibes the problem that in theory high-dimensional data conatins more information, but in practice this is not the case. Higher dimensional data often contains more noise and redundancy and therefore adding more information does not benefit for downstream analysis steps.

Not all genes are informative and are important for the task of cell type clustering based on their expression profiles. We already aimed to reduce the dimensionality of the data with feature selection, as a next step one can further reduce the dimensions of single-cell RNA-seq data with dimensionality reduction algorithms. These algorithms are an important step during preprocessing to reduce the data complexity and for visualization. Several dimensionality reduction techniques have been developed and used for single-cell data analysis.

Xing et al. compared in an independent comparison the stability, accuracy and computing cost of 10 different dimensionality reduction methods [Xiang et al., 2021]. They propose to use t-distributed stochastic neighbor embedding (t-SNE) as it yielded the best overall performance. Uniform manifold approximation and projection (UMAP) showed the highest stability and separates best the original cell populations. An additional dimensionality reduction worth mentioning in this context is principal component analysis (PCA) which is still widely used.

Generally, t-SNE and UMAP are very robust and mostly equivalent if specific choices for the initialization are selected [Kobak and Berens, 2019].

We will use a normalized representation of the dataset for dimensionality reduction and visualization, specifically the shifted logarithm.

In [ ]:
for key in adatas.keys():
    adatas[key].X = adatas[key].layers["log1p_norm"]

In [ ]:
#PCA


In [ ]:
for key in adatas.keys():
    # setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
    adatas[key].var["highly_variable"] = adatas[key].var["highly_deviant"]
    sc.pp.pca(adatas[key], svd_solver="arpack", use_highly_variable=True)

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pl.pca_scatter(adatas[key], color="total_counts")

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pl.pca_variance_ratio(adatas[key], log=True)

In [ ]:
#tsne

In [ ]:
for key in adatas.keys():
    print(key)
    sc.tl.tsne(adatas[key], use_rep="X_pca")
    sc.pl.tsne(adatas[key], color="total_counts")

In [ ]:
#Umap

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pp.neighbors(adatas[key])
    sc.tl.umap(adatas[key])

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pl.umap(adatas[key], color="total_counts")

In [ ]:
##QC metrics on UMAP

In [ ]:
sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)

In [ ]:
for key in adatas.keys():
    print(key)
    with rc_context({'figure.figsize': (7,7)}):
        sc.pl.umap(
            adatas[key],
            color=["total_counts", "pct_counts_mt"],
            vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
            size=10, cmap='RdYlBu_r'
        )
        sc.pl.umap(
            adatas[key],
            color=["scDblFinder_score", "scDblFinder_class"],
            size=10, cmap='RdYlBu_r'
        )

In [ ]:
#Subset doublets

In [ ]:
for key in adatas.keys():
    print(f"Total number of cells: {adatas[key].n_obs}")
    adatas[key] = adatas[key][adatas[key].obs.scDblFinder_class != 2].copy()
    print(f"Number of cells after filtering of low quality cells: {adatas[key].n_obs}")

In [ ]:
print( adatas[key].obs.scDblFinder_class)
print(key)

In [ ]:
for key in adatas.keys():
# Define a boolean mask to filter out doublets
    is_singlet = adatas[key].obs.scDblFinder_class == 'singlet'  # Replace with the actual annotation name

# Invert the boolean mask to keep non-doublets

# Create a new AnnData object containing only non-doublets
    adatas[key] = adatas[key][is_singlet].copy()


In [ ]:
for key in adatas.keys():
    print(key)
    with rc_context({'figure.figsize': (7,7)}):
        sc.pl.umap(
            adatas[key],
            color=["scDblFinder_score", "scDblFinder_class"],
            size=10, cmap='RdYlBu_r'
        )

In [ ]:
print(adatas[key])



In [ ]:
#Clustering

In [ ]:
for key in adatas.keys():
    sc.tl.leiden(adatas[key], key_added="leiden_res0_1", resolution=0.1)
    sc.tl.leiden(adatas[key], key_added="leiden_res0_2", resolution=0.2)
    sc.tl.leiden(adatas[key], key_added="leiden_res0_25", resolution=0.25)
    sc.tl.leiden(adatas[key], key_added="leiden_res0_5",  resolution=0.5)
    sc.tl.leiden(adatas[key], key_added="leiden_res1_0",  resolution=1.0)
    sc.tl.leiden(adatas[key], key_added="leiden_res1_5",  resolution=1.5)

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pl.umap(
        adatas[key],
        color=["leiden_res0_1", "leiden_res0_2", "leiden_res0_25", "leiden_res0_5", "leiden_res1_0", "leiden_res1_5"],
        legend_loc="on data",
        ncols = 3, legend_fontoutline=2
    )

In [ ]:
#Cell annotation

##Marker genes

In [ ]:
cell_cyle_genes = {
"G1":["Ccne1","Pttg1"],
"G1/S":["Slbp","Cdca7","Ung","Cdc6","Pcna","Mcm5","Orc1","Dtl"],
"S":["Rrm2","E2f8","Atad2"],
"G2":["Ccnf","Cdca8","Hjurp","Cdk1","Top2a","Kif23"],
"G2/M":["Kpna2","Bub1b","Tacc3"],
"M":["Plk1","Mapk13","Aurka","Tpx2"]
}

In [ ]:
#Marker genes from Wirka2019NatMed, Table S1 (Top 20 genes per group), from mouse aortic root and ascending aorta

In [ ]:
marker_genes_Wirka2019 = {
"SMCs1":["Acta2","Tagln","Tnnt2","Tpm2","Myl9","Cnn1","Tpm1","Dstn","Sost","Lmod1","Myh11","Csrp2","Csrp1","Actg2","Ppp1r14a","Tcap","Hspb1","Cald1","Hspb6","Mylk"],
"SMCs2":["Acta2","Cnn1","Myl9","Myh11","Tagln","Tpm2","Sost","Flna","Itga8","Ppp1r14a","Dstn","Csrp1","Col18a1","Csrp2","Sh3bgr","Mylk","Myl6","Actn1","Tpm1","Nov"],
"Fibroblasts1":["Pi16","Clec3b","Gpx3","Serping1","Cygb","Dpep1","Smoc2","Gsn","C3","Igfbp4","Pcolce","Fbln1","Serpinf1","Entpd2","Ly6a","Abca8a","Cxcl1","Mfap5","Col3a1","Dpt"],
"Fibroblasts2":["Comp","Chad","Wif1","1500015O10Rik","Thbs1","Fmod","Angptl7","Prg4","Fxyd6","Prelp","Dkk3","Ccdc80","Meox1","Dcn","Ptn","Npy","Aldh1a1","Dpt","Itih5","Abi3bp"],
"Fibromyocytes":["Spp1","Ibsp","Fn1","Col2a1","Lcn2","Lum","Timp1","Tnfrsf11b","Mgp","1500015O10Rik","Clu","Frzb","Serpina3n","Serpine2","Sncg","Vcam1","Rarres1","Ltbp2","Lgals3","Crispld1","Crlf1","Anxa8","Bgn","Acan","Tnc","Scara3","Ank","C3","Sdc1","Cx3cl1"],
"Macrophages":["Cd74","Lyz2","C1qa","C1qb","H2-Aa","H2-Ab1","C1qc","H2-Eb1","Fcer1g","Wfdc17","Ctss","Cxcl2","Ccl8","Tyrobp","Pf4","Ccl6","Cd14","Ccl9","Il1b","Ctsc"],
"ECs1":["Ctla2a","Pecam1","Ly6c1","Cldn5","Gpihbp1","Egfl7","Cd36","Esam","Emcn","Icam2","Vwf","Cdh5","Tspan13","Eng","Slc9a3r2","Sdpr","Ptprb","Ecscr","Ramp2","Flt1"],
"ECs2":["Ccl21a","Nts","Lrg1","Cldn5","Mmrn1","Ecscr","Lbp","Clca3a1","Apold1","Egfl7","Pard6g","Ctla2a","Galnt18","Flt4","Emcn","Gpm6a","Gngt2","Pvrl2","Prox1","Icam2"],
"Pericytes":["Colec11","Ndufa4l2","Vtn","Higd1b","Ifitm1","Notch3","Nrip2","Olfr558","Ephx3","Casq2","Myo1b","Adamts4","Rgs5","Ngf","Adap2os","S1pr3","Rgs16","Igfbp5","Enpep","Parm1"],
"Epithelial cells":["Upk3b","Msln","Fxyd3","Nkain4","Krt19","Gpm6a","Krt7","Myl7","Upk1b","Cldn15","Rspo1","C2","Krt8","Wt1","Tmem151a","Dmkn","Lrrn4","Chst4","Dpp4","Slc39a8"],
"Neurons":["Plp1","Kcna1","Gfra3","Egfl8","Sostdc1","Mbp","Gpr37l1","Cadm4","Nrn1","Lgi4","Chl1","Tspan15","Fabp7","Adam23","Gfap","Cdh19","Plekhb1","Cmtm5","Mpz","Ttyh1"]
}

In [ ]:
#Marker genes from Zernecke2022CardiovascRes, Fig 4, positive and negative markers, from full mouse aorta

In [ ]:
marker_genes_Zernecke2022 = {
"DCs": ["Dpp4","Itgax"],
"DCs negative": ["Adgre1","C5ar1"],
"cDCs1": ["Clec9a","Xcr1","Irf8","Naaa"],
"cDCs1 negative": ["Itgam","Sirpa"],
"cDCs2": ["Itgam","Sirpa"],
"cDCs2 negative": ["Clec9a","Xcr1","Cd209a","Clec10a","Klrd1"],
"mature DCs":["Ccr7","Fscn1","Ccl17","Ccl22","Tnfrsf4","Il4i1","Cd200","Cd274","Cd40","Cd80","Cd86"],
    
# TBC...
    
}

In [ ]:
#Marker genes from Anisimov2023NatCardiovascRes, full text, from full mouse aorta

In [ ]:
marker_genes_Anisimov2023 = {
"Hemoglobin": ["Hba-a1","Hba-a2"],
"Macrophages": ["Trem2","Lyz2","Cd68","Lyve1","Cxcl2","Ccl2"],
"Monocytes": ["Ccr2","Lyz1"],
"SMCs": ["Acta2","Tagln"],
"ECs": ["Cdh5","Pecam1"],
"NK/T cells": ["Cd3d","Cd8b1","Nkg7"],
"ECs": ["Cdh5","Pecam1"],
"B cells": ["Cd79a","Igkc","Ighm","Cd79b"],
"Leukocytes": ["Cd45"],
"Fibroblasts": ["Dcn","Sperinf1","Fbn1","ApoE","Igfbp5","Pdgfra","Serpinf1","Clec3b","Tie2","Mfap4"],
"T cells": ["Ccl5","Ccr9","Cd8","Cxcr6"],
"Proliferating immune cells": ["Top2a"],
"Mast cells": ["Kit"],
"DCs": ["Cd209a"]
}

In [ ]:
#Sample1

In [ ]:
key = list(adatas.keys())[0]
key

In [ ]:
adatatemp = adatas[key].copy()

Cell Cycle
Score groups:

In [ ]:
def sort_and_score(adatatemp, marker_genes):
    
    marker_genes_in_data = {}
    for ct, markers in marker_genes.items():
        markers_found = []
        for marker in markers:
            if marker in adatatemp.var.index:
                markers_found.append(marker)
        if markers_found:
            marker_genes_in_data[ct] = markers_found

    for key in marker_genes_in_data.keys():
        sc.tl.score_genes(adatatemp, marker_genes_in_data[key], score_name=key)
        
    return adatatemp, marker_genes_in_data

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, cell_cyle_genes)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    sort_order=True
    )

Individual marker genes

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False, cmap="Reds",
    )
    print("\n\n\n")  # print white space for legibility

Cell types
Score groups:

Anisimov 2023

In [ ]:
# load above
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Anisimov2023)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.3, 
    size= 10, sort_order=True
    )

Individual marker genes:

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False,
        cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
        size = 8
    )
    print("\n\n\n")  # print white space for legibility

Wirka 2019

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Wirka2019)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    size=10, sort_order=True
    )

Individual marker genes:

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False,
        cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
        size=8,
    )
    print("\n\n\n")  # print white space for legibility

In [ ]:
del adatatemp

Cluster DEGs

In [ ]:
res = "res0_2" # select most sensible res

In [ ]:
sc.tl.rank_genes_groups(
    adatas[key], groupby="leiden_"+res, method="wilcoxon", key_added="dea_leiden_"+res
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adatas[key], groupby="leiden_"+res, standard_scale="var", n_genes=10, key="dea_leiden_"+res
)

In [ ]:
with rc_context({'figure.figsize': (9,9)}):
    sc.pl.umap(adatas[key], color=["leiden_"+res], legend_loc="on data", size=14)

Filter for cluster-specific DEGs and re-plot:

In [ ]:
sc.tl.filter_rank_genes_groups(
    adatas[key],
    min_in_group_fraction=0.2,
    max_out_group_fraction=0.2,
    key="dea_leiden_"+res,
    key_added="dea_leiden_"+res+"_filtered",
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adatas[key],
    groupby="leiden_"+res,
    standard_scale="var",
    n_genes=5,
    key="dea_leiden_"+res+"_filtered",
)

sample 2

In [ ]:
key = list(adatas.keys())[1]
key

In [ ]:
adatatemp = adatas[key].copy()

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, cell_cyle_genes)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    sort_order=True
    )

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False, cmap="Reds",
    )
    print("\n\n\n")  # print white space for legibility

In [ ]:
# load above
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Anisimov2023)
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    size= 8, sort_order=True
    )

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False,
        cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
        size = 8
    )
    print("\n\n\n")  # print white space for legibility

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Wirka2019)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    size=8, sort_order=True
    )

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False,
        cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
        size=8,
    )
    print("\n\n\n")  # print white space for legibility

In [ ]:
del adatatemp

Cluster DEGs

In [ ]:
res = "res0_5" # select most sensible res

In [ ]:
sc.tl.rank_genes_groups(
    adatas[key], groupby="leiden_"+res, method="wilcoxon", key_added="dea_leiden_"+res
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adatas[key], groupby="leiden_"+res, standard_scale="var", n_genes=10, key="dea_leiden_"+res
)

In [ ]:
with rc_context({'figure.figsize': (9,9)}):
    sc.pl.umap(adatas[key], color=["leiden_"+res], legend_loc="on data", size=14)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adatas[key],
    min_in_group_fraction=0.2,
    max_out_group_fraction=0.2,
    key="dea_leiden_"+res,
    key_added="dea_leiden_"+res+"_filtered",
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adatas[key],
    groupby="leiden_"+res,
    standard_scale="var",
    n_genes=5,
    key="dea_leiden_"+res+"_filtered",
)


In [ ]:
save_dir = main_dir+"snRNAseq_Python_env/"

# Annotate Clusters

In [ ]:
#HFD
HFD_annotation = {
    "0": "SMCs",
    "1": "SMCs",
    "2": "ECs",
    "3": "Ms",
    "4": "FBs",
    "5": "ACs",
    "6" : "SMCs",
    "7" : "FBMCs",
    "8" : "NA",
    "9" : "NA2"}

In [ ]:
key = list(adatas.keys())[0]
key


In [ ]:
print(adatas[key].obs
     )

In [ ]:
adatas[key].obs["cell_type"] = adatas[key].obs.leiden_res0_25.map(HFD_annotation)

In [ ]:
sc.pl.umap(adatas[key], color=["cell_type", "leiden_res0_25"])

In [ ]:
adata_HFD_sub=adatas[key].copy()
print(key)

In [ ]:
adata_HFD_sub = adata_HFD_sub[adata_HFD_sub.obs['cell_type'].isin(["ACs","ECs","FBMCs","FBs","Ms","SMCs"])]

In [ ]:
sc.pl.umap(adata_HFD_sub, color=["cell_type", "leiden_res0_25"])

In [ ]:
#HFD L1
HFD_L1_annotation = {
    "0": "SMCs",
    "1": "SMCs",
    "2": "ECs",
    "3": "Ms",
    "4": "FBs",
    "5": "ACs",
    "6": "SMCs",
    "7": "FBMCs",
    "8": "SMCs",
    "9": "ECs2",
    "10": "ECs",
    "11": "SMCs",
   
}

In [ ]:
key = list(adatas.keys())[1]
key


In [ ]:
adatas[key].obs["cell_type"] = adatas[key].obs.leiden_res0_5.map(HFD_L1_annotation)

In [ ]:
print(adatas[key].obs)
print(key)

In [ ]:
sc.pl.umap(adatas[key], color=["cell_type", "leiden_res0_5","n_genes_by_counts"])

In [ ]:
for key in adatas.keys():
    print(key)
    print(adatas[key])

In [ ]:
save_dir = main_dir+"snRNAseq_Python_env/"

# Perform merging of the two datasets

In [ ]:
label_key = "cell_type"
batch_key = "batch"

In [ ]:
import scanpy.external as sce

In [ ]:
key = list(adatas.keys())[1]
key
print(key)
adatas[key].obs
adatas[key].obs["batch"] = key
adatas[key].obs

In [ ]:
key = list(adatas.keys())[0]
key
print(key)
adatas[key].obs
adatas[key].obs["batch"] = key
adatas[key].obs

In [ ]:
key = list(adatas.keys())[1]
key
#adatas[key].obs['names'] = adatas[key].obs['names'].astype(str)
adatas[key].write_h5ad(save_dir+f"annotated_{key}_data.h5ad")
print(key)
print(adatas[key])

In [ ]:
for key in adatas.keys():
    
    adatas[key] = sc.read_h5ad(save_dir+
    f"annotated_{key}_data.h5ad"
    )
    print(key)
    print(adatas[key])

In [ ]:
for key in adatas.keys():
    print(key)
    sc.pl.umap(adatas[key], color=["cell_type", "leiden_res0_5", "n_genes_by_counts",'pct_counts_mt', 'total_counts_ribo' ])
    
    


In [ ]:
#merge
merged = adatas[list(adatas.keys())[1]].concatenate(adatas[list(adatas.keys())[0]])


In [ ]:
merged.write_h5ad(save_dir+"annotated_merged_data.h5ad")

In [ ]:
annotated_merged_data=sc.read_h5ad(save_dir+"annotated_merged_data.h5ad"
    )
combine_datasets = annotated_merged_data.copy()

In [ ]:
#merge
print(combine_datasets)



In [ ]:
sc.pl.umap(combine_datasets, color = ["batch", "cell_type","leiden_res0_1","n_genes_by_counts"])

In [ ]:
combine_datasets.obs['batch'] = combine_datasets.obs['batch'].replace("1", "HFD")

In [ ]:
print(combine_datasets.obs['batch']
     )

In [ ]:
combine_datasets.write_h5ad(save_dir+"combine_data_afer_QC.h5ad")

In [ ]:

filename =main_dir+'snRNAseq_Python_env/Snapshot_Wspc_snrnaseq_QC.pkl'

print(filename)

In [ ]:
#!pip install dill
import dill                            #pip install dill --user

dill.dump_session(filename)

# and to load the session again:
#dill.load_session(filename)

ECs2 seem to be doublets or low quilty cells